# Predicción de Borough de Destino — NYC Yellow Taxi 2015-01

**Tarea 1 — Sistemas Urbanos Inteligentes (ICT3115, 2026-1)**  
**Problema:** Clasificación multiclase — predecir el borough de destino (`DO_Borough_id`)  
**Features:** tiempo de pickup + zona de origen (PULocationID) + metadata Census ACS del barrio de origen  
**Autores:** Nicolás Herrera y Vincent Metzker

> **Nota:** todas las features usadas son información disponible en el momento del pickup (sin leakage de datos del viaje en curso).

Este notebook cubre:
1. Carga de datos y preparación
2. MLP base (sin embeddings)
3. MLP con embeddings (PULocationID como embedding)
4. Análisis de sobreajuste + regularización
5. AutoEncoder + MLP
6. Análisis crítico final

## 0. Setup

In [ ]:
import random
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
SEED = 22041991

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print('Dispositivo:', DEVICE)

DATA_DIR = '../data'

## 1. Carga y preparación de datos

### 1.1 Cargar muestra

In [ ]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'trips_sample_with_cats.parquet'))
print(f'Shape cargado: {df.shape}')
df.head(3)

### 1.2 Agregar metadata Census + DO_Borough_id

Si el parquet fue guardado antes de correr la sección 4.7 del notebook de EDA,
los campos census y borough no estarán presentes. Este bloque los añade si faltan.

In [ ]:
if 'PU_Borough_id' not in df.columns:
    print('Columnas census no encontradas — ejecutando join...')
    lookup = pd.read_csv(os.path.join(DATA_DIR, 'taxi_zone_lookup_enriched.csv'))
    drop_cols = ['Zone', 'service_zone']

    pu_lookup = (lookup.drop(columns=drop_cols, errors='ignore')
                 .add_prefix('PU_').rename(columns={'PU_LocationID': 'PULocationID'}))
    do_lookup = (lookup.drop(columns=drop_cols, errors='ignore')
                 .add_prefix('DO_').rename(columns={'DO_LocationID': 'DOLocationID'}))

    df = df.merge(pu_lookup, on='PULocationID', how='left')
    df = df.merge(do_lookup, on='DOLocationID', how='left')

    BOROUGH_MAP = {'Manhattan': 0, 'Brooklyn': 1, 'Queens': 2,
                   'Bronx': 3, 'Staten Island': 4, 'EWR': 5}
    df['PU_Borough_id'] = df['PU_Borough'].map(BOROUGH_MAP).fillna(6).astype(np.int8)
    df['DO_Borough_id'] = df['DO_Borough'].map(BOROUGH_MAP).fillna(6).astype(np.int8)
    df.drop(columns=['PU_Borough', 'DO_Borough'], inplace=True, errors='ignore')

    census_cols = [c for c in df.columns if c.startswith(('PU_', 'DO_'))
                   and c not in ('PU_Borough_id', 'DO_Borough_id')]
    df[census_cols] = df[census_cols].fillna(0)
    print('Join completado:', df.shape)
else:
    print('Columnas census ya presentes.')

### 1.3 Filtrar clases target y definir features

In [ ]:
# ── Target ────────────────────────────────────────────────────────────────────
TARGET = 'DO_Borough_id'

# Solo 5 boroughs principales (eliminar EWR=5 y Unknown=6, son < 0.1% del total)
df = df[df[TARGET] <= 4].copy()
df[TARGET] = df[TARGET].astype(np.int64)

BOROUGH_NAMES = {0: 'Manhattan', 1: 'Brooklyn', 2: 'Queens', 3: 'Bronx', 4: 'Staten Island'}

# ── Features numéricas (disponibles en tiempo de pickup) ─────────────────────
NUM_FEATS = [
    'passenger_count',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'PU_Population', 'PU_MedianHouseholdIncome', 'PU_HousingUnits',
    'PU_Transport_CarAlone_pct', 'PU_Transport_Carpool_pct',
    'PU_Transport_PublicTransit_pct', 'PU_Transport_Walked_pct',
    'PU_Transport_Bicycle_pct', 'PU_Transport_Other_pct',
    'PU_Transport_WorkFromHome_pct',
]

# ── Features categóricas ──────────────────────────────────────────────────────
# Para MLP base: PU_Borough_id + hora/día como ordinales
CAT_FEATS_BASE = ['VendorID', 'RatecodeID', 'PU_Borough_id',
                  'is_weekend', 'is_rush_hour', 'is_night', 'is_morning', 'is_afternoon']

# Para MLP con embeddings: PULocationID (alta cardinalidad) + borough + hora/día
CAT_FEATS_EMB = {
    'PULocationID':  265,   # num clases → tamaño vocab
    'PU_Borough_id': 5,
    'RatecodeID':    6,
    'VendorID':      2,
}

print(f'Filas tras filtro de target: {len(df):,}')
print(f'Clases target: {df[TARGET].value_counts().sort_index().to_dict()}')
print(f'Num features: {len(NUM_FEATS)}')

## 2. Análisis del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = df[TARGET].value_counts().sort_index()
labels = [BOROUGH_NAMES[i] for i in vc.index]

axes[0].bar(labels, vc.values, color='steelblue', edgecolor='none')
axes[0].set_title('Distribución de DO_Borough_id')
axes[0].set_ylabel('Nº viajes')
axes[0].tick_params(axis='x', rotation=20)

axes[1].pie(vc.values, labels=labels, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(vc)))
axes[1].set_title('Proporción por borough de destino')

plt.tight_layout()
plt.show()

print('\nDesbalance de clases:')
print((vc / vc.sum() * 100).round(2).to_frame('pct_%'))

## 3. Split y preprocesamiento

In [ ]:
# ── Copias para cada variante ─────────────────────────────────────────────────
df_onehot = df.copy()   # MLP base (sección 4)
df_emb    = df.copy()   # MLP con embeddings (sección 5)
df_ae     = df.copy()   # AutoEncoder (sección 7)

# ── Índices de split estratificados ──────────────────────────────────────────
idx = np.arange(len(df))
tv_idx, test_idx = train_test_split(idx, test_size=0.15,
                                     stratify=df[TARGET].values, random_state=SEED)
train_idx, val_idx = train_test_split(tv_idx, test_size=0.1765,
                                       stratify=df.iloc[tv_idx][TARGET].values, random_state=SEED)

print(f'Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')

### 3.1 Preprocesamiento para MLP base (one-hot)

In [ ]:
# One-hot sobre categóricas de baja cardinalidad
cat_onehot_cols = ['VendorID', 'RatecodeID', 'PU_Borough_id']
df_onehot_enc = pd.get_dummies(df_onehot, columns=cat_onehot_cols, drop_first=False)

# Features binarias/ordinales ya están como int — se incluyen directamente
binary_feats = ['is_weekend', 'is_rush_hour', 'is_night', 'is_morning', 'is_afternoon',
                'hour', 'day_of_week', 'month']

onehot_cols = [c for c in df_onehot_enc.columns
               if c.startswith(('VendorID_', 'RatecodeID_', 'PU_Borough_id_'))]

features_base = NUM_FEATS + binary_feats + onehot_cols

X_base = df_onehot_enc[features_base].astype(np.float32).values
y_base = df_onehot_enc[TARGET].values

# Scaler ajustado solo en train
scaler_base = StandardScaler()
X_base[train_idx] = scaler_base.fit_transform(X_base[train_idx])
X_base[val_idx]   = scaler_base.transform(X_base[val_idx])
X_base[test_idx]  = scaler_base.transform(X_base[test_idx])

N_CLASSES = int(df[TARGET].nunique())
print(f'Features baseline: {len(features_base)} | Clases: {N_CLASSES}')

### 3.2 DataLoaders baseline

In [ ]:
BATCH_SIZE = 256

def make_loader(X, y, idx, shuffle=True):
    Xt = torch.tensor(X[idx], dtype=torch.float32)
    yt = torch.tensor(y[idx], dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader_base = make_loader(X_base, y_base, train_idx, shuffle=True)
val_loader_base   = make_loader(X_base, y_base, val_idx,   shuffle=False)
test_loader_base  = make_loader(X_base, y_base, test_idx,  shuffle=False)

xb, yb = next(iter(train_loader_base))
print('Batch X:', xb.shape, '| Batch y:', yb.shape)

## 4. MLP Base (sin embeddings)

Las variables categóricas de baja cardinalidad (VendorID, RatecodeID, PU_Borough_id) se representan con **one-hot encoding**.  
PULocationID se excluye aquí para evitar 265 dimensiones adicionales — en la sección 5 se incorpora como embedding.

In [ ]:
class MLPBase(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128, 64), dropout=0.3, n_classes=5):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model_base = MLPBase(input_dim=X_base.shape[1], n_classes=N_CLASSES).to(DEVICE)
print(model_base)
print(f'\nParámetros: {sum(p.numel() for p in model_base.parameters()):,}')

In [ ]:
# ── Funciones de entrenamiento / evaluación ───────────────────────────────────

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        logits = model(Xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def get_preds(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for Xb, yb in loader:
        logits = model(Xb.to(DEVICE))
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred)

def plot_curves(train_vals, val_vals, metric='Loss', title=''):
    epochs = range(1, len(train_vals) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_vals, label='Train')
    plt.plot(epochs, val_vals, label='Validación')
    plt.xlabel('Época'); plt.ylabel(metric)
    plt.title(title or metric); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
EPOCHS = 60
LR     = 1e-3

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model_base.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

hist_base = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, EPOCHS + 1):
    tl, ta = train_epoch(model_base, train_loader_base, criterion, optimizer)
    vl, va = eval_epoch(model_base,  val_loader_base,   criterion)
    scheduler.step()

    hist_base['train_loss'].append(tl); hist_base['val_loss'].append(vl)
    hist_base['train_acc'].append(ta);  hist_base['val_acc'].append(va)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | train_loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f}')

In [ ]:
plot_curves(hist_base['train_loss'], hist_base['val_loss'], 'Cross-Entropy Loss', 'MLP Base — Curvas de pérdida')
plot_curves(hist_base['train_acc'],  hist_base['val_acc'],  'Accuracy',            'MLP Base — Curvas de accuracy')

In [ ]:
y_true_base, y_pred_base = get_preds(model_base, test_loader_base)

print('=== Evaluación MLP Base (test) ===')
print(f'Accuracy:          {accuracy_score(y_true_base, y_pred_base):.4f}')
print(f'Balanced Accuracy: {balanced_accuracy_score(y_true_base, y_pred_base):.4f}')
print()
print(classification_report(y_true_base, y_pred_base,
                             target_names=list(BOROUGH_NAMES.values())))

cm = confusion_matrix(y_true_base, y_pred_base)
disp = ConfusionMatrixDisplay(cm, display_labels=list(BOROUGH_NAMES.values()))
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de confusión — MLP Base')
plt.tight_layout(); plt.show()

## 5. MLP con Embeddings

La variable `PULocationID` tiene 265 valores únicos (zonas TLC).  
En el MLP base eso sería 265 dimensiones one-hot; aquí se reemplaza por una **capa Embedding** que aprende una representación densa de dimensión `emb_dim`.  
También se agregan embeddings para `PU_Borough_id`, `RatecodeID` y `VendorID`.

In [ ]:
# ── Preparar datos para modelo con embeddings ─────────────────────────────────
# Categóricas como índices enteros (0-based, corregidos)
for col, n_cats in CAT_FEATS_EMB.items():
    df_emb[col] = df_emb[col].clip(0, n_cats - 1).astype(np.int64)

# Reindexar PULocationID a 0-based (el lookup va de 1 a 265)
df_emb['PULocationID'] = (df_emb['PULocationID'] - 1).clip(0, 264).astype(np.int64)

X_num_emb = df_emb[NUM_FEATS + ['hour', 'day_of_week', 'month',
                                  'is_weekend', 'is_rush_hour', 'is_night',
                                  'is_morning', 'is_afternoon']].astype(np.float32).values
X_cat_emb = df_emb[list(CAT_FEATS_EMB.keys())].astype(np.int64).values
y_emb     = df_emb[TARGET].values

# Scaler en numéricas
scaler_emb = StandardScaler()
X_num_emb[train_idx] = scaler_emb.fit_transform(X_num_emb[train_idx])
X_num_emb[val_idx]   = scaler_emb.transform(X_num_emb[val_idx])
X_num_emb[test_idx]  = scaler_emb.transform(X_num_emb[test_idx])

print(f'X_num: {X_num_emb.shape} | X_cat: {X_cat_emb.shape}')

In [ ]:
class EmbDataset(Dataset):
    def __init__(self, X_num, X_cat, y, idx):
        self.X_num = torch.tensor(X_num[idx], dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat[idx], dtype=torch.long)
        self.y     = torch.tensor(y[idx],     dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X_num[i], self.X_cat[i], self.y[i]

train_ds_emb = EmbDataset(X_num_emb, X_cat_emb, y_emb, train_idx)
val_ds_emb   = EmbDataset(X_num_emb, X_cat_emb, y_emb, val_idx)
test_ds_emb  = EmbDataset(X_num_emb, X_cat_emb, y_emb, test_idx)

train_loader_emb = DataLoader(train_ds_emb, batch_size=BATCH_SIZE, shuffle=True)
val_loader_emb   = DataLoader(val_ds_emb,   batch_size=512,        shuffle=False)
test_loader_emb  = DataLoader(test_ds_emb,  batch_size=512,        shuffle=False)

In [ ]:
def emb_dim(n_cats):
    """Regla de dedo: min(50, (n_cats + 1) // 2)""    return min(50, (n_cats + 1) // 2)

class MLPWithEmb(nn.Module):
    def __init__(self, num_dim, cat_vocab, hidden_dims=(256,128,64), dropout=0.3, n_classes=5):
        super().__init__()
        # Embeddings por variable categórica
        self.embs = nn.ModuleList([
            nn.Embedding(n_cats, emb_dim(n_cats)) for n_cats in cat_vocab
        ])
        total_emb = sum(emb_dim(n) for n in cat_vocab)
        input_dim = num_dim + total_emb

        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        emb_out = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
        x = torch.cat([x_num] + emb_out, dim=1)
        return self.net(x)

cat_vocabs = list(CAT_FEATS_EMB.values())
model_emb = MLPWithEmb(num_dim=X_num_emb.shape[1], cat_vocab=cat_vocabs,
                        n_classes=N_CLASSES).to(DEVICE)
print(model_emb)
print(f'\nParámetros: {sum(p.numel() for p in model_emb.parameters()):,}')

In [ ]:
def train_epoch_emb(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xn, xc, yb in loader:
        xn, xc, yb = xn.to(DEVICE), xc.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xn, xc)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch_emb(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for xn, xc, yb in loader:
        xn, xc, yb = xn.to(DEVICE), xc.to(DEVICE), yb.to(DEVICE)
        logits = model(xn, xc)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def get_preds_emb(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for xn, xc, yb in loader:
        logits = model(xn.to(DEVICE), xc.to(DEVICE))
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred)

In [ ]:
EPOCHS_EMB = 60
optimizer_emb = optim.Adam(model_emb.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_emb = optim.lr_scheduler.StepLR(optimizer_emb, step_size=20, gamma=0.5)

hist_emb = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, EPOCHS_EMB + 1):
    tl, ta = train_epoch_emb(model_emb, train_loader_emb, criterion, optimizer_emb)
    vl, va = eval_epoch_emb(model_emb, val_loader_emb, criterion)
    scheduler_emb.step()

    hist_emb['train_loss'].append(tl); hist_emb['val_loss'].append(vl)
    hist_emb['train_acc'].append(ta);  hist_emb['val_acc'].append(va)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | train_loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f}')

In [ ]:
plot_curves(hist_emb['train_loss'], hist_emb['val_loss'], 'Cross-Entropy Loss', 'MLP Embeddings — Curvas de pérdida')
plot_curves(hist_emb['train_acc'],  hist_emb['val_acc'],  'Accuracy',            'MLP Embeddings — Curvas de accuracy')

In [ ]:
y_true_emb, y_pred_emb = get_preds_emb(model_emb, test_loader_emb)

print('=== Evaluación MLP con Embeddings (test) ===')
print(f'Accuracy:          {accuracy_score(y_true_emb, y_pred_emb):.4f}')
print(f'Balanced Accuracy: {balanced_accuracy_score(y_true_emb, y_pred_emb):.4f}')
print()
print(classification_report(y_true_emb, y_pred_emb,
                             target_names=list(BOROUGH_NAMES.values())))

cm = confusion_matrix(y_true_emb, y_pred_emb)
disp = ConfusionMatrixDisplay(cm, display_labels=list(BOROUGH_NAMES.values()))
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de confusión — MLP con Embeddings')
plt.tight_layout(); plt.show()

### 5.1 Visualización del espacio de embeddings de PULocationID

In [ ]:
from sklearn.decomposition import PCA

# Extraer pesos del embedding de PULocationID (índice 0 en self.embs)
pu_weights = model_emb.embs[0].weight.data.cpu().numpy()  # (265, emb_dim)

# Reducir a 2D con PCA
pca = PCA(n_components=2, random_state=SEED)
pu_2d = pca.fit_transform(pu_weights)

# Colorear por borough (del lookup)
lookup_plot = pd.read_csv('../data/taxi_zone_lookup_enriched.csv')
BOROUGH_MAP_INV = {'Manhattan': 0, 'Brooklyn': 1, 'Queens': 2, 'Bronx': 3, 'Staten Island': 4}
lookup_plot['borough_id'] = lookup_plot['Borough'].map(BOROUGH_MAP_INV)

# Alinear (LocationID 1-265 → índice 0-264)
colors = lookup_plot.set_index('LocationID')['borough_id'].reindex(range(1, 266)).fillna(-1).values

fig, ax = plt.subplots(figsize=(9, 7))
palette = sns.color_palette('muted', 5)
for bid, bname in BOROUGH_NAMES.items():
    mask = colors == bid
    ax.scatter(pu_2d[mask, 0], pu_2d[mask, 1], label=bname,
               color=palette[bid], alpha=0.7, s=40)

ax.set_title('PCA del espacio de embedding de PULocationID (265 zonas)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(title='Borough')
plt.tight_layout(); plt.show()

## 6. Análisis de sobreajuste y regularización

In [ ]:
# Comparación visual de curvas de ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_base['train_loss'], label='Base train', lw=1.5)
axes[0].plot(hist_base['val_loss'],   label='Base val',   lw=1.5, ls='--')
axes[0].plot(hist_emb['train_loss'],  label='Emb train',  lw=1.5)
axes[0].plot(hist_emb['val_loss'],    label='Emb val',    lw=1.5, ls='--')
axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Época'); axes[0].legend()

axes[1].plot(hist_base['train_acc'], label='Base train', lw=1.5)
axes[1].plot(hist_base['val_acc'],   label='Base val',   lw=1.5, ls='--')
axes[1].plot(hist_emb['train_acc'],  label='Emb train',  lw=1.5)
axes[1].plot(hist_emb['val_acc'],    label='Emb val',    lw=1.5, ls='--')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Época'); axes[1].legend()

plt.suptitle('Comparación de curvas train/val — Base vs Embeddings')
plt.tight_layout(); plt.show()

# TODO (Actividad 4):
# - Analizar si existe sobreajuste (brecha train/val creciente)
# - Probar al menos una técnica adicional: dropout más alto, weight decay, early stopping
# - Comentar el efecto observado

## 7. AutoEncoder + MLP

In [ ]:
# TODO (Actividad 5): Implementar AutoEncoder
#
# Estructura sugerida:
#   1. Entrenar AE sobre las variables numéricas (NUM_FEATS + binary) → reconstrucción MSE
#   2. Congelar encoder → usar bottleneck como entrada a un MLP clasificador
#   3. Comparar métricas vs. MLP base y MLP con embeddings
#
# Referencia de arquitectura del Taller_T1 (sección 5):
#
# class AutoEncoder(nn.Module):
#     def __init__(self, input_dim, hidden=[128,64], bottleneck=32, dropout=0.2):
#         super().__init__()
#         # Encoder
#         enc_layers = []
#         prev = input_dim
#         for h in hidden:
#             enc_layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]
#             prev = h
#         enc_layers.append(nn.Linear(prev, bottleneck))
#         self.encoder = nn.Sequential(*enc_layers)
#         # Decoder (espejo)
#         dec_layers = []
#         prev = bottleneck
#         for h in reversed(hidden):
#             dec_layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]
#             prev = h
#         dec_layers.append(nn.Linear(prev, input_dim))
#         self.decoder = nn.Sequential(*dec_layers)
#
#     def forward(self, x):
#         z = self.encoder(x)
#         return self.decoder(z), z

print('Sección pendiente — implementar en Actividad 5')

## 8. Análisis crítico final

In [ ]:
# Tabla resumen comparativa (completar con resultados reales)
results = {
    'Modelo': ['MLP Base', 'MLP Embeddings', 'AE + MLP'],
    'Accuracy (test)':          [accuracy_score(y_true_base, y_pred_base),
                                  accuracy_score(y_true_emb,  y_pred_emb),
                                  None],
    'Balanced Acc (test)':      [balanced_accuracy_score(y_true_base, y_pred_base),
                                  balanced_accuracy_score(y_true_emb,  y_pred_emb),
                                  None],
}

pd.DataFrame(results).set_index('Modelo').round(4)

### Puntos a desarrollar:

1. **MLP base vs MLP con embeddings**: ¿Mejoró la accuracy? ¿Qué clases beneficiaron más del embedding de PULocationID?
2. **Visualización de embeddings**: ¿El espacio aprendido de las 265 zonas refleja geografía real? ¿Zonas del mismo borough aparecen agrupadas?
3. **Sobreajuste y regularización**: ¿Hubo brecha creciente entre train y val? ¿Qué técnica se aplicó y cuál fue el efecto?
4. **AutoEncoder**: ¿El bottleneck captura representaciones útiles para clasificación? ¿Mejora o empeora vs. el MLP base?
5. **Limitación principal**: la metadata Census es a nivel borough, no de zona — todas las zonas del mismo borough comparten exactamente los mismos valores, lo que limita su poder discriminativo para predecir el borough de destino.